# Agents I: Reasoning Models and the Agent Loop


Lecture 14 derived the reinforcement-learning recipe that turns a chat model
into a reasoning model, and lecture 15 showed what a reasoning model costs to
serve. This lecture is about what to *do* with one. The first part is a short,
practical account of reasoning models: what the word means, when such a model
is the right tool, the four ways to build one, and the DeepSeek-R1 family as
the worked example. The second part moves from the model to the software
around it: the agent loop, the ReAct pattern that formalized it, and the six
components of a coding harness, because in 2026 most of the useful work a
model does happens inside one. The closing section applies both halves to our
own subject: agents that run machine-learning experiments, which is how the
midterm project works.

The first part follows Sebastian Raschka's
[*Understanding Reasoning LLMs* (Feb 2025)](https://magazine.sebastianraschka.com/p/understanding-reasoning-llms)
and the second his
[*Components of a Coding Agent* (Apr 2026)](https://magazine.sebastianraschka.com/p/components-of-a-coding-agent);
the figures credited to him come from those essays. UDL does not cover this
material; where a method has a primary paper the lecture cites it and uses its
terms. The quizzable calculation is the prompt-caching cost of an agent run in
section 6.


## Today's roadmap

**Part 1: reasoning models**

1. What "reasoning" means, and when a reasoning model is the right tool
2. What lecture 14 gave us; the DeepSeek-R1 family
3. Four ways to build a reasoning model, with inference-time scaling in depth
4. Reasoning on a budget, and reasoning practice in 2026

**Part 2: the agent loop and the coding harness**

5. LLM vs. reasoning model vs. agent; the loop; ReAct; harness vs. model
6. Six components of a coding harness
7. Agents for ML research: autoresearch and OpenResearch, and the midterm project
8. Summary; pointer to lecture 22


# Part 1: Reasoning models

A reasoning model is a chat model that has been trained to write out
intermediate steps before answering. This section says what that buys, what
it costs, and when to reach for one.


## Defining "reasoning"

![Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/regular-vs-reasoning-output.png)


No agreed definition; a working one:

> **Reasoning** = answering questions that need multi-step generation with intermediate steps.

| Query | Reasoning needed? |
|---|---|
| "Capital of France?" | No: lookup |
| "60 mph for 3 hours, how far?" | Yes: relate three quantities |
| "Prove $\sqrt{2}$ is irrational." | Yes: multi-step proof |

The steps can be **visible** (the model writes its work) or **hidden** (a thinking block the user does not see, as in OpenAI's o1 and most 2026 "thinking" modes). Any LLM can be prompted to show steps; a *reasoning model* does it by default, because it was trained to.



## When is a reasoning model the right tool?

![Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/reasoning-strengths-weaknesses.png)


**Good fit:** math and competition problems, logic puzzles, multi-file coding
tasks, planning under constraints: anything where a wrong intermediate step
makes the answer wrong.

**Poor fit:** summarization, translation, fact retrieval, short creative
writing: anything a single pass already does well.

The price: **more tokens, more latency, more cost per answer**, and a tendency to
**overthink** easy prompts ([Chen et al. 2024](https://arxiv.org/abs/2412.21187)
measure o1-style models spending many times the tokens of a plain model on
"2 + 3 = ?").

This trade-off is why every provider ships a regular and a "thinking" variant,
or one model with a switch.


## What lecture 14 gave us

Four things to carry in from the RL lecture, so this one does not repeat them:

- **A language model is a policy**, and a response is a trajectory; the
  policy-gradient theorem says how to push probability toward responses that
  score well.
- **GRPO** samples a group of $G$ responses per prompt and uses the group mean
  as the baseline, so no value network is needed; the advantage is the same for
  every token of a response.
- **Verifiable rewards (RLVR)**: where a program can check the answer (math
  with a reference, code with tests, a format check), use it instead of a
  learned reward model. DeepSeek-R1-Zero's reward is accuracy plus format,
  nothing else.
- **The result**: response length and accuracy grow together under RL with no
  reward for length, with the caveats that the base model already had most of
  the behaviour and that RLVR mostly sharpens what the base could sample.


Everything below treats these as known and asks the engineering questions:
which recipe, at what cost, for which task.


## Case study: the DeepSeek-R1 family

![Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/deepseek-r1-training-pipeline.png)


Three models from one base ([DeepSeek-V3](https://arxiv.org/abs/2412.19437), 671B total, 37B active), each a different lesson:

| Model | Lesson |
|---|---|
| **R1-Zero** | reasoning can emerge from **pure RL**, no SFT |
| **R1** | **SFT + RL** gives the strongest, most usable model |
| **R1-Distill** | small models catch up by **SFT on R1's outputs** |

The [technical report](https://arxiv.org/abs/2501.12948) (Jan 2025; the
*Nature* version is arXiv v2) is the blueprint most 2025–26 open reasoning
models followed, including the Qwen3 pipeline we meet later.


# The four ways to build a reasoning model

DeepSeek's three models illustrate three of the four general approaches; the
fourth needs no training at all. We take them in order of how much they change
the model.


## Four approaches

1. **Inference-time scaling**: spend more compute per query; no weight updates
2. **Pure RL**: RL with verifiable rewards on a base model, no SFT (R1-Zero)
3. **SFT + RL**: cold-start SFT, then RL, then more SFT and RL (R1)
4. **Pure SFT / distillation**: fine-tune a small model on a reasoner's outputs (R1-Distill)

They are complementary: a production system typically trains with 3, ships
small variants with 4, and serves with some of 1.


## Approach 1: inference-time scaling

**Idea:** more compute *at inference* for a better answer, with the weights
fixed. Three flavors:

- **Chain-of-thought prompting**: "think step by step"
  ([Wei et al. 2022](https://arxiv.org/abs/2201.11903);
  zero-shot: [Kojima et al. 2022](https://arxiv.org/abs/2205.11916))
- **Self-consistency / majority vote**: sample $N$ answers, return the most
  common ([Wang et al. 2022](https://arxiv.org/abs/2203.11171))
- **Search with a verifier**: best-of-$N$, beam search or tree search scored
  by a reward model

![Zero-shot chain-of-thought prompting. Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/zero-shot-cot-prompting.png)

Why does self-consistency help, and when can it hurt? Model each sample as
correct with probability $p$, independently. For a yes/no answer, a majority
of $N$ votes (odd $N$) is correct with probability
$P(\text{Binomial}(N, p) > N/2)$, which climbs to 1 as $N$ grows when
$p > 0.5$ and falls to 0 when $p < 0.5$: voting amplifies whatever the model
already does most often. With many possible answers the condition is weaker:
the correct answer only has to be the *most common* one, so votes can rescue a
model with $p < 0.5$ whose wrong answers are spread out, and not one whose
wrong answers agree. This is one reading of Snell et al.'s negative bars on the
hardest questions (next slides): when the model's modal answer is wrong, more
samples make it more confidently wrong. Change `ps` or the wrong-answer split
and rerun; the dots are a Monte Carlo check of the exact curves.

In [ ]:
#| code-fold: true
#| fig-cap: "Majority-vote accuracy against the number of samples $N$. Left: a binary answer, exact binomial curves (lines) and 20,000 simulated questions (dots). Right: plurality vote over five possible answers with the correct one sampled 40% of the time, when the wrong 60% is concentrated on one answer or spread over four."
#| fig-alt: "Two panels against N from 1 to 31. Left: for p of 0.9, 0.7 and 0.55 the majority-vote accuracy rises toward 1, faster for larger p; for p of 0.45 and 0.3 it falls toward 0. Right: with the correct answer at 40%, plurality voting rises toward 1 when the wrong answers are spread 15% each over four, and falls toward 0 when one wrong answer gets 60%."
import numpy as np
import matplotlib.pyplot as plt
from math import comb

rng = np.random.default_rng(0)
ps = [0.9, 0.7, 0.55, 0.45, 0.3]       # per-sample probability of the right answer
Ns = np.arange(1, 32, 2)               # odd N, so a binary vote has no ties
n_questions = 20_000

def majority_exact(N, p):
    return sum(comb(N, j) * p**j * (1 - p) ** (N - j) for j in range(N // 2 + 1, N + 1))

def plurality_sim(N, probs):
    """Answer 0 is correct; ties are broken uniformly at random."""
    counts = rng.multinomial(N, probs, size=n_questions)
    top = counts == counts.max(axis=1, keepdims=True)
    return np.mean(top[:, 0] / top.sum(axis=1))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
colors = ["#008300", "#1baf7a", "#2a78d6", "#eb6834", "#e34948"]
ax = axes[0]
for p, c in zip(ps, colors):
    ax.plot(Ns, [majority_exact(N, p) for N in Ns], color=c, lw=2, label=f"$p$ = {p}")
    sim = [np.mean(rng.binomial(N, p, n_questions) > N / 2) for N in Ns]
    ax.plot(Ns, sim, "o", color=c, ms=3.5)
ax.set_xlabel("samples $N$")
ax.set_ylabel("majority-vote accuracy")
ax.set_title("binary answer", fontsize=10)
ax.legend(fontsize=8.5, frameon=False, loc="center right")

ax = axes[1]
splits = {"wrong 60% spread over 4 answers": [0.4, 0.15, 0.15, 0.15, 0.15],
          "wrong 60% on one answer": [0.4, 0.6, 0, 0, 0]}
for (label, probs), c in zip(splits.items(), ["#2a78d6", "#e34948"]):
    ax.plot(Ns, [plurality_sim(N, probs) for N in Ns], color=c, lw=2, marker="o", ms=3.5, label=label)
ax.axhline(0.4, color="#8a8984", lw=0.8, ls="--")
ax.set_xlabel("samples $N$")
ax.set_title("five answers, correct one sampled 40% of the time", fontsize=10)
ax.legend(fontsize=8.5, frameon=False, loc="center right")
for ax in axes:
    ax.set_ylim(0, 1.02)
    ax.grid(color="#dcdcd8", lw=0.6)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

for p in [0.55, 0.7]:
    print(f"p = {p}: one sample {p:.2f}, vote of 5 {majority_exact(5, p):.3f}, "
          f"vote of 31 {majority_exact(31, p):.3f}")

## Search-based inference scaling

![Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/search-based-inference-methods.png)


A **process reward model (PRM)** scores *partial* chains, an **outcome reward
model (ORM)** only final answers.

- **Best-of-$N$**: $N$ full answers, keep the top-scored
- **Beam search**: keep the top-$k$ partial chains at each step
- **Lookahead / MCTS**: roll out a few steps before scoring a branch

The R1 report lists PRM-guided search and MCTS under *unsuccessful attempts*
(section 4.2): training a good PRM is hard, the search space of tokens is
enormous, and the model learned to exploit the PRM. What they got instead is
**implicit** test-time scaling: R1 simply writes longer chains than V3.


## Compute-optimal test-time scaling (Snell et al. 2024)

![Figure 1 of [Snell et al. 2024](https://arxiv.org/abs/2408.03314). Left: MATH accuracy vs. generation budget for sequential revisions; "compute-optimal" picks sequential or parallel per question. Right: accuracy gain from test-time compute, relative to a pretrained model 14× larger, in a FLOPs-matched comparison, split by question difficulty and by the ratio of inference to pretraining tokens.](figs/21-Reasoning-Models-and-Agents/snell-compute-optimal-revisions.png)

- **Question:** given a fixed budget of generated tokens, how should it be
  spent: more parallel samples, or sequential revisions of one answer, or
  search against a PRM?
- **Finding 1:** the right choice depends on the question. Easy questions gain
  from *revising* one answer; hard ones from *parallel* search. Choosing per
  question (by a predicted difficulty) is "compute-optimal" and beats
  best-of-$N$ by **more than 4×** in efficiency.
- **Finding 2:** on questions where the small model has a non-trivial success
  rate, test-time compute lets it **outperform a model 14× larger** at matched
  FLOPs. On the hardest questions, and when inference tokens far exceed
  pretraining tokens, pretraining the bigger model wins (the negative bars).


The FLOPs-matched comparison counts $6N$ FLOPs per pretraining token and $2N$
per inference token (lecture 12's rules), so "the ratio of inference to
pretraining tokens" on the horizontal axis is what decides whether buying more
inference or a bigger pretrained model is the better use of the same compute.
The practical reading for 2026: test-time compute is cheapest to buy when the
base model is already close, and is not a substitute for capability it lacks.


## Approach 2: pure RL (R1-Zero)

![DeepSeek-R1-Zero development process. Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/deepseek-r1-zero-process.png)


Start from the V3 *base* model, skip SFT, run GRPO with the accuracy and format
rewards of lecture 14 on reasoning prompts.

> Reasoning behaviours (re-checking, backtracking, long chains) emerged from the
> reward alone; nobody wrote them into training data.

What it did not give: readable outputs. R1-Zero mixes languages and ignores
formatting, which is the problem approach 3 fixes.

Lecture 14 has the training curves and the two caveats (the base model already
had the behaviour; RL sharpens more than it discovers).


## Approach 3: SFT + RL (R1, the flagship recipe)

![DeepSeek-R1 development process. Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/deepseek-r1-process.png)


Four stages ([DeepSeek-AI 2025](https://arxiv.org/abs/2501.12948), section 2.3):

1. **Cold start:** a few thousand long, readable chains (filtered R1-Zero
   outputs, cleaned by hand) fine-tune V3 so RL starts from legible behaviour
2. **Reasoning RL:** accuracy + format + a **language-consistency** reward
   (fraction of target-language words in the chain) to stop language mixing;
   it costs a little accuracy and the authors kept it for readability
3. **Rejection sampling + SFT:** about 600k reasoning samples from the RL
   checkpoint, plus about 200k non-reasoning samples (writing, QA) so the model
   stays a general assistant
4. **RL for all scenarios:** verifiable rewards on reasoning data, a reward
   model for helpfulness and harmlessness on the rest


## R1 vs. R1-Zero: does the extra work pay off?

![Benchmark comparison of OpenAI o1 and DeepSeek-R1. Figure: Raschka, *Understanding Reasoning LLMs* (2025), from Tables 2 and 4 of the R1 report, arXiv v1 (Jan 2025).](figs/21-Reasoning-Models-and-Agents/o1-vs-r1-benchmarks.png)

Read the table, not the headline. In the arXiv v1 numbers, R1 beats R1-Zero on
AIME (79.8 vs. 71.0), MATH-500 (97.3 vs. 95.9), LiveCodeBench (65.9 vs. 50.0)
and CodeForces (2029 vs. 1444); on **GPQA Diamond R1-Zero is slightly ahead**
(73.3 vs. 71.5). The SFT stages buy readability, generality and most of the
coding gain, not a uniform improvement. R1-Zero is the scientific result; R1
is the product. The Nature version (v2, Jan 2026) reports R1-Zero AIME 77.9
and GPQA 75.8, which narrows the AIME gap to 1.9 points; lecture 14 uses v2.


## Approach 4: pure SFT (distillation)

![DeepSeek-R1-Distill development process. Figure: Raschka, *Understanding Reasoning LLMs* (2025).](figs/21-Reasoning-Models-and-Agents/deepseek-r1-distill-process.png)


Take the ~800k SFT samples from stage 3 and fine-tune **Qwen 2.5 and Llama 3
bases, 1.5B to 70B**, with ordinary cross-entropy. No RL.

> This is **sequence-level distillation** (lecture 15), not logit matching: the
> student only sees the teacher's sampled outputs, which is why it works through
> an API.

Two motives: **efficiency** (a 7B reasoner runs on a laptop) and the research
question *how far does SFT alone go?*

A variant worth knowing: *journey learning*
([Qin et al. 2024](https://arxiv.org/abs/2410.18982)) fine-tunes on the whole
trial-and-error trace, wrong turns and corrections included, rather than only
on clean correct solutions, and gets some of RL's self-correction from SFT
alone.


## Distillation results

![Benchmark comparison of distilled and non-distilled models. Figure: Raschka, *Understanding Reasoning LLMs* (2025), from Table 5 of the R1 report.](figs/21-Reasoning-Models-and-Agents/distillation-benchmarks.png)

- **R1-Distill-Qwen-32B** is above o1-mini on AIME (72.6 vs. 63.6) and
  MATH-500 (94.3 vs. 90.0); **R1-Distill-Llama-70B** is close to it.
- The same report ran R1-Zero's pure-RL recipe directly on Qwen-32B and got a
  *worse* model than distilling into it (lecture 15 states the 32B result):
  **small base, distill; large base, RL**.
- The catch: distillation needs a stronger teacher to exist, and the student
  inherits the teacher's long chains, so it is cheaper per token but not per
  answer.


## When to use which approach

| Approach | Strength | Weakness |
|---|---|---|
| **1. Inference-time scaling** | no training; works on any model | pays per query, forever |
| **2. Pure RL** | clean scientific signal; no demonstrations | needs a strong base; outputs unpolished |
| **3. SFT + RL** | best quality and usability | the most expensive pipeline |
| **4. Distillation** | cheap; deployable on one GPU | ceilinged by the teacher |

In practice (DeepSeek, Qwen3, the closed labs): **3 to make the flagship, 4 to
make the small ones, 1 at serving time when the budget allows.**


# Reasoning on a budget, and reasoning in 2026

You do not need thousands of GPUs to try the ideas above. Two January-2025
projects showed what a few hundred dollars buys, and the 2026 practice of
thinking budgets shows how the cost side is managed in production.


## Two recipes at lab scale: Sky-T1 (about \$450) and TinyZero (under \$30)

**Sky-T1** ([NovaSky, UC Berkeley, Jan 2025](https://novasky-ai.github.io/posts/sky-t1/)):

- **Base:** Qwen2.5-32B-Instruct
- **Data:** 17k traces from QwQ-32B-Preview, kept only when the final answer
  verified (**rejection sampling**), then reformatted for readability
- **Training:** SFT only, no RL; about **\$450** of GPU time
- **Result:** matches o1-preview on the math and coding benchmarks they report

Approach 4 with a different teacher. The lesson the authors drew: a small,
*curated* dataset beats a large noisy one.


**TinyZero** ([Jiayi Pan et al., Berkeley, Jan 2025](https://github.com/Jiayi-Pan/TinyZero)):

- **Base:** Qwen2.5-3B
- **Task:** the *Countdown* arithmetic game (reach a target from given
  numbers), which has a trivial verifier
- **Training:** R1-Zero-style RL, under **\$30**
- **Observation:** the 3B model learned to try, check and discard candidates
  inside its `<think>` block

Approach 2 at toy scale: the finding replicates, on a narrow domain. Not a
general reasoner, and exactly the kind of run you can do on one SCC GPU
(lecture 14, section 7).


## Reasoning practice in 2026: thinking budgets

![Figure 1 of [Yang et al. 2025](https://arxiv.org/abs/2505.09388), the Qwen3 technical report: the post-training pipeline. Stages 1–2 are the R1 recipe; stage 3 fuses thinking and non-thinking modes into one model.](figs/21-Reasoning-Models-and-Agents/qwen3-post-training-pipeline.png)


Long chains cost output tokens, the expensive kind (lecture 15). 2026 systems
make the length a dial:

- **Hybrid models.** Qwen3 is one model with a thinking and a non-thinking
  mode, and a **thinking budget**: when the chain hits the user's limit the
  harness inserts a stop-thinking instruction and the model answers from what
  it has; accuracy rises smoothly with the budget (Qwen3 report, Figure 2).
- **Budget forcing** is the same trick as a research result:
  [s1](https://arxiv.org/abs/2501.19393) appends "Wait" to extend thinking or a
  terminator to end it.
- **Adaptive thinking.** API providers have moved from a fixed token budget to
  the model deciding how much to think, steered by an *effort* level
  ([Anthropic](https://platform.claude.com/en/docs/build-with-claude/extended-thinking),
  [OpenAI](https://platform.openai.com/docs/guides/reasoning),
  [Google](https://ai.google.dev/gemini-api/docs/thinking)).


Why this matters for cost: a 10,000-token thinking chain in front of a
200-token answer is a 50× multiplier on output tokens for that request, and
output tokens are priced several times above input tokens on every major API.
An agent that makes a hundred calls per task inherits that multiplier on every
call. Lecture 15's cost-per-million-tokens formula applies directly: the
thinking budget is the knob that trades accuracy against it.


## Prompting a reasoning model

Reasoning models behave differently from chat models. From the R1 report
(the report's prompting-engineering discussion and the GitHub usage
recommendations) and the Qwen3 report:

1. **Zero-shot beats few-shot.** Examples in the prompt *hurt* R1; describe the
   problem and the output format, then stop.
2. **Keep the prompt in one language**, or the chain will mix.
3. **Do not ask for brevity on hard problems**; set a thinking budget instead if
   cost matters.
4. **Verify the final answer, not the chain.** The chain is a sample, not a
   proof; the format reward makes it *look* like one.


Now put one of these models inside a loop.


# Part 2: the agent loop and the coding harness

The second part moves from the model to the software around it. The claim to
test is that, for a coding task, the loop and its supporting machinery (the
harness) determine as much of the outcome as the model does.


## Why talk about agents?

![Claude Code, Codex CLI and Raschka's Mini Coding Agent. Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/coding-agent-clis.png)


Much of the practical progress of 2024–26 was not better models. It was better
**scaffolding around the same models**: tools, context management, memory, a
control loop.

The same model feels dramatically more capable in Claude Code or Codex CLI
than in a chat window.

> The model is the engine. The harness is the car.

Coding is where this shows most, because the task needs files, tests and
tracebacks, and **a next-token model cannot run anything**.


## LLM vs. reasoning model vs. agent

![Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/llm-reasoning-agent-relationship.png)

| Concept | What it is |
|---|---|
| **LLM** | a next-token model |
| **Reasoning model** | an LLM trained to produce intermediate steps and check itself |
| **Agent** | a control loop that uses a model **+ tools + memory + environment feedback** |
| **Agent harness** | the software scaffold: prompts, tools, state, control flow, permissions |
| **Coding harness** | a harness specialized for software engineering (Claude Code, Codex, OpenCode) |

## The agent loop

An agent repeats one cycle inside an environment:

```text
        ┌─── observe ──► gather information (files, tool output, user)
        │
        ▼
      inspect ──► reason about it
        │
        ▼
      choose ──► pick the next action
        │
        ▼
        act ──► execute it (tool call, edit, command)
        │
        └────► back to observe
```

Stop when the goal is met, the model says it is done, or the budget (steps,
tokens, dollars) runs out.

This is the "agentic trajectory" of lecture 14's agentic-RL section, seen from the
software side: the policy's tokens are the actions, tool outputs are the
observations.


## The formal origin: ReAct (Yao et al. 2022)

![The ReAct loop and a shortened HotpotQA trace. Source: instructor figure after Figure 1 of [Yao et al. 2022](https://arxiv.org/abs/2210.03629).](figs/21-Reasoning-Models-and-Agents/react-loop.png)

- **ReAct** ("Reasoning + Acting") has the model emit free-text *thoughts* and
  *actions* in alternation; the environment returns an *observation* after each
  action, which goes back into the prompt.
- Thoughts keep the plan on track and let the model recover from a bad
  observation; actions let it look things up instead of hallucinating.
- Results in the paper: on HotpotQA and FEVER with a Wikipedia API, fewer
  hallucinated facts than chain-of-thought alone; on ALFWorld and WebShop,
  **+34 and +10 points absolute** success over imitation and RL baselines, from
  one or two in-context examples.
- Every 2026 harness is this loop with structured tool calls instead of
  free-text actions, a budget, and a lot of engineering around the prompt.


## A coding harness has three layers

![Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/coding-harness-three-layers.png)


1. **Model family**: the LLM or reasoning model (the engine)
2. **Agent loop**: observe, inspect, choose, act
3. **Runtime supports**: repo context, tools, permissions, caching, memory,
   sandboxing

> Same model + better harness = a different product.

Everything in layer 3 is ordinary software. That is the point: it is the part
you can build, inspect and test.


## The claim: harness > model (often)

Frontier models from different vendors are close in raw capability.
Raschka's claim (April 2026), verbatim:

> *"This is speculative, but I suspect that if we dropped one of the latest,
> most capable open-weight LLMs, such as GLM-5, into a similar harness, it
> could likely perform on par with GPT-5.4 in Codex or Claude Opus 4.6 in
> Claude Code."*

- The harness is frequently the distinguishing factor between products.
- Caveat: vendors do **harness-specific post-training** (lecture 14's agentic
  RL: multi-turn rollouts in *their* tool environment), so the model and the
  harness co-evolve; OpenAI ships a general and a `codex` variant of the same
  model.
- For you: a stronger harness is cheaper to get than a stronger model, and an
  open-weight model in a good harness is a real option.


# The six components of a coding harness

Raschka breaks a coding harness into six components, and ships a small
reference implementation
([Mini Coding Agent](https://github.com/rasbt/mini-coding-agent)) that has all
six in a few hundred lines. Each component answers a problem the model alone
cannot.


## Why coding needs a harness, and the six components

![Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/harness-features-overview.png)


"Fix the failing tests" needs: find the right files among thousands, look up
where `encode_batch` is defined, apply a diff rather than rewrite, run the
tests, read the traceback, try again. None of that is next-token prediction.

1. **Live repo context**
2. **Prompt shape and cache reuse**
3. **Structured tools, validation and permissions**
4. **Context reduction and output management**
5. **Transcripts, memory and resumption**
6. **Delegation to bounded subagents**


## Component 1: live repo context

![Workspace summary combined with the user request. Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/workspace-summary-context.png)


"Fix the tests" cannot be answered in a vacuum. Before the first model call the
harness gathers:

- git state: branch, status, recent commits
- project docs: `AGENTS.md`, `CLAUDE.md`, `README`
- layout, and the build and test commands

**Without:** "I'll run `pytest`" (wrong; this repo uses `make test`).\
**With:** "`AGENTS.md` says tests run via `make test`. Running."

This is why a one-page instructions file in the repo root became an ecosystem
convention: it is the cheapest context there is.


## Component 2: prompt shape and cache reuse

![Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/stable-prefix-dynamic-tail.png)


Naive: concatenate everything and send it every turn. But most of it does not
change between turns.

| Part | Changes? |
|---|---|
| system instructions | no |
| tool descriptions | no |
| workspace summary | rarely |
| working memory, recent transcript | yes |
| latest user turn | every time |

So assemble the prompt as a **stable prefix** followed by a **dynamic tail**,
and let the provider **cache the KV tensors of the prefix** (lecture 15's
prefix caching, exposed at the API). Long sessions get large latency and cost
wins from this one layout decision.


## What prompt caching is

**Problem:** every call re-runs prefill over the whole prompt: long system
prompt, tool schemas, and a transcript that grows with every tool result.

**Mechanism:** the provider keeps the **KV cache** (the keys and values every
layer computed, lecture 15) for a prompt *prefix*; a later call with the
identical prefix skips that prefill and continues from the cached state.


- **Exact-prefix match on tokens**: one changed character invalidates
  everything after it
- **Time-limited**: entries expire after minutes unless refreshed
- **Priced**: a cache *write* costs more than plain input, a cache *read* much
  less ([Anthropic](https://platform.claude.com/docs/en/build-with-claude/prompt-caching):
  writes 1.25× for a 5-minute entry, reads 0.1×; up to 4 explicit breakpoints.
  [OpenAI](https://platform.openai.com/docs/guides/prompt-caching) caches
  automatically above 1,024 tokens;
  [Google](https://ai.google.dev/gemini-api/docs/caching) exposes a cache handle)


## Prompt caching: the quizzable calculation

A 20-step agent task: stable prefix $P = 20{,}000$ tokens; each step appends
$t = 1{,}000$ tokens of tool result and model output. Count input tokens in
*full-price units* (write $= 1.25$, read $= 0.1$).

- **No cache.** Step $k$ sends $P + (k-1)t$ tokens at full price:
  $$\sum_{k=1}^{20}\big(P + (k-1)t\big) = 20P + 190\,t = 400{,}000 + 190{,}000 = 590{,}000.$$
- **Cache.** Step 1 writes $P$: $1.25 \times 20{,}000 = 25{,}000$. Step $k \ge 2$
  reads everything sent so far, $P + (k-2)t$, and writes the new $t$:
  $$\sum_{k=2}^{20}\Big(0.1\,\big(P + (k-2)t\big) + 1.25\,t\Big) = 0.1\,(19P + 171\,t) + 19 \times 1.25\,t = 55{,}100 + 23{,}750 = 78{,}850.$$
- **Total** $25{,}000 + 78{,}850 = 103{,}850$ units: about **5.7× cheaper** on
  input, and prefill time drops in proportion. Output tokens are unchanged.


**Design rule that follows:** stable content first (system prompt, tools, repo
summary, then history, then the new turn); never a timestamp, request ID or
"current time" near the top; append to the transcript, never rewrite it.


The same sum, step by step, as a calculator. Change the prefix, the tokens
per step, the number of steps or the two price multipliers (OpenAI's
automatic cache, for instance, charges nothing extra to write) and rerun. The
last lines show how the saving grows with the length of the run; for long
runs both sums are dominated by the $t K^2/2$ term, so the ratio approaches
$1/0.1 = 10\times$ and never passes it.

In [ ]:
P = 20_000          # stable prefix: system prompt, tool schemas, workspace summary
t = 1_000           # tokens appended per step: model output + clipped tool result
K = 20              # steps in the task
write, read = 1.25, 0.10   # cache write and read prices, as multiples of plain input

def costs(P, t, K, write=write, read=read):
    no_cache = sum(P + (k - 1) * t for k in range(1, K + 1))
    cached = write * P                                   # step 1 writes the prefix
    for k in range(2, K + 1):
        cached += read * (P + (k - 2) * t) + write * t   # read what was sent, write the new t
    return no_cache, cached

no_cache, cached = costs(P, t, K)
print(f"no cache:   {no_cache:>9,.0f} token-units  (= K*P + t*K(K-1)/2 = {K * P + t * K * (K - 1) // 2:,})")
print(f"with cache: {cached:>9,.0f} token-units  (step 1 writes {write * P:,.0f}, steps 2..K cost {cached - write * P:,.0f})")
print(f"saving: {no_cache / cached:.2f}x on input tokens")

print()
for steps in [1, 2, 5, 10, 20, 50, 100]:
    a, b = costs(P, t, steps)
    print(f"K = {steps:>3}: {a:>11,.0f} vs {b:>9,.0f}  ->  {a / b:5.2f}x")

## Component 3: structured tools and the tool-use flow

![Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/tool-use-flow.png)


A chat model *suggests* `pytest tests/test_auth.py -v` in prose. An agent
*runs* it, through a **structured tool call** with a schema, not free text:

`list_files(path)`, `read_file(path, start, end)`, `search(pattern)`,
`run_shell(cmd)`, `write_file(path, contents)`, `apply_diff(path, diff)`

1. Model emits a structured action
2. Harness **validates**: known tool? valid arguments? path inside the
   workspace? needs approval?
3. Ask the user if it does
4. **Execute**, capture output
5. Feed the **clipped** result back into the loop


## Why validation and permissions matter

![Tool-call approval request in the Mini Coding Agent. Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/tool-approval-request.png)


The harness restricts the model on purpose:

- rejects malformed actions
- sandboxes paths to the repository
- gates dangerous actions (`rm -rf`, force push, network) behind approval
- limits blast radius: no `sudo`, no writes outside the workspace

Less freedom makes the agent **more** useful: the user can leave it running.
The tool list is also the agent's **action space**, which is what lecture 14's
agentic RL trains over and what lecture 22's tool protocols (MCP) standardize.


## Component 4: context reduction

![Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/context-compaction.png)


Every step adds a file read, a log, a search result or a diff. Unchecked, the
context fills in a few turns, cost scales with it, and quality degrades on long
contexts even when they fit.

**Clipping:** truncate long outputs (head and tail, elide the middle) so no one
result dominates the budget.

**Transcript reduction:** recent events rich, older ones summarized;
deduplicate repeated reads of the same file.

> *"A lot of apparent 'model quality' is really context quality."*

If an agent re-reads the same file five times or forgets something from twenty
turns ago, that is usually a context problem, not a model problem.


## Component 5: structured session memory

![Full transcript and working memory. Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/transcript-and-working-memory.png)


Two layers of state with different jobs:

| Layer | Format | Role |
|---|---|---|
| **Full transcript** | JSONL, append-only | the durable record; resume after a crash |
| **Working memory** | small structured JSON | current task, key files, notes, open questions; *edited*, not appended |

```json
{"current_task": "fix failing auth tests",
 "key_files": ["src/auth.py", "tests/test_auth.py"],
 "recent_notes": ["test_login_timeout needs a mock"],
 "open_questions": ["keep backward compat?"]}
```

Transcript = *what happened*; memory = *what matters now*. Close the laptop,
reopen tomorrow: the agent loads both. Lecture 22 covers long-term memory
across sessions.


## Component 6: bounded subagents

![A subagent inherits context but runs inside tighter boundaries. Figure: Raschka, *Components of a Coding Agent* (2026).](figs/21-Reasoning-Models-and-Agents/bounded-subagent.png)


Side questions ("which file defines `encode_batch`?", "why does this test
fail?") can run inline, polluting the main context, or be **delegated** to a
subagent that returns one focused answer.

The tricky design problem, in Raschka's words, is "not just how to spawn a
subagent but also how to bind one".

Typical bounds: read-only by default, scoped context, a tool whitelist, a step
or time budget, no recursive spawning. The subagent reports and disappears.

Multi-agent *orchestration* beyond this, and when it helps, is lecture 22.


## The harness in 2026 products

Four coding harnesses you will meet, named only; check each one's documentation
for current features:

- [**Claude Code**](https://docs.claude.com/en/docs/claude-code/overview)
  (Anthropic): terminal agent; also packaged as an SDK
- [**Codex CLI**](https://github.com/openai/codex) (OpenAI): open-source
  terminal agent
- [**Cursor**](https://cursor.com): an editor with the agent built in
- [**OpenCode**](https://opencode.ai): open-source, MIT-licensed, works with
  many model providers including local ones

What to look for in any of them, by component: a repo instructions file
(`CLAUDE.md` / `AGENTS.md`), prompt caching in the token report, an approval or
permission mode, a compaction step when the context fills, session resume, and
subagents. You can map every one of them to a slide above.


# Agents for ML research

The closing section applies the two halves of the lecture to our own subject.
An agent that runs machine-learning experiments is an agent loop with a
verifiable reward, a constrained action space, and a harness that keeps state,
which is exactly what the midterm project asks you to set up and then to
judge.


## autoresearch: one file, five minutes, one number

![The autoresearch loop. Source: instructor figure after the [karpathy/autoresearch](https://github.com/karpathy/autoresearch) README.](figs/21-Reasoning-Models-and-Agents/autoresearch-loop.png)

- A nanochat-style GPT on one GPU. The agent may edit **only `train.py`**
  (model, Muon + AdamW optimizer, training loop): a constrained action space.
- Each experiment trains for **exactly 5 minutes** of wall clock and is scored
  on **validation bits per byte**, which is vocabulary-independent so
  architecture changes compare fairly: a verifiable reward.
- **About 12 experiments an hour, about 100 overnight.** The human writes
  `program.md`, the research brief, and reads the log in the morning.
- It is approach 2 of part 1 and the loop of part 2 in one script: the agent
  proposes, the environment scores, the log is the memory.


## OpenResearch: the harness for research agents

[alphaXiv/OpenResearch](https://github.com/alphaXiv/OpenResearch) is "the
local-first harness and workspace for research agents": the six components,
specialized for science instead of software.

| Harness component | OpenResearch's version |
|---|---|
| repo context | **literature**: search over alphaXiv, bioRxiv and PubMed; hypotheses tied to prior work |
| tools | **compute backends**: local, SSH, Slurm, Kubernetes, Modal, HF Jobs and others |
| memory | an **experiment ledger** in a local SQL database: code, logs, artifacts, so the agent learns from past runs |
| model | works with Claude Code, Codex, OpenCode, Cursor, or a local model |

The Slurm and SSH modes are how it would talk to the SCC. It is a desktop app
under active development; for the project, any tracker that records every run
(including a spreadsheet) is acceptable.


## Agent as instrument, student as scientist

The midterm project: set up an autoresearch-style loop on your own small
transformer on one SCC GPU, write the brief, let it run. Then do **the science
the agent cannot**:

- pick the three changes that mattered, and **re-run each with several seeds**:
  a 5-minute run's noise is as large as many of its "improvements"
- explain *why* each helped, with a mechanism, not a metric
- write a two-page **research note**, by the humans
- defend it in an **individual oral**


**Guardrails** (the syllabus's AI-use rules, made concrete): the agent's full
log is submitted; every kept change needs a seed-checked ablation; the note is
human-written; the oral is individual.

**Where these loops fail** (lecture 22 goes deeper): reward hacking the metric,
overfitting the validation set, changes that only work at 5 minutes, and
confounds such as batch size vs. learning rate.


# Summary

One line per idea; the recap deck uses these.


## Summary

- A reasoning model is an LLM trained to produce intermediate steps by default; use it for multi-step, verifiable tasks and not for lookup, summarization or anything a single pass does well, because it costs more tokens, latency and money per answer.
- Four ways to build one: inference-time scaling (no training), pure RL with verifiable rewards (R1-Zero), SFT + RL (R1, the flagship recipe), and distillation by SFT on a reasoner's outputs (R1-Distill); flagships use 3, small models 4, serving adds 1.
- Inference-time scaling: chain-of-thought, self-consistency, and search against a verifier (best-of-$N$, beam, lookahead); DeepSeek found PRM search and MCTS not worth it and got implicit scaling from longer chains instead.
- Snell et al. 2024: spend test-time compute per question (revisions for easy, parallel search for hard); compute-optimal allocation is over 4× more efficient than best-of-$N$, and on questions a small model can partly solve it beats a model 14× larger at matched FLOPs, but not on the hardest ones.
- R1 beats R1-Zero on AIME, MATH-500, LiveCodeBench and CodeForces but not on GPQA Diamond (arXiv v1 numbers; the Nature v2 narrows the AIME gap to 1.9 points); the SFT stages buy readability, generality and coding, not a uniform gain.
- R1-Distill-Qwen-32B beats o1-mini on AIME and MATH-500; pure RL on the same 32B does worse than distilling into it: small base, distill; large base, RL.
- Sky-T1 (17k rejection-sampled traces, SFT, about \$450) and TinyZero (3B model, Countdown, RL, under \$30) show both recipes at lab scale.
- 2026 practice makes chain length a dial: hybrid thinking/non-thinking models with a thinking budget (Qwen3), budget forcing (s1), and adaptive thinking with an effort level at the APIs; zero-shot prompts, one language, verify the answer not the chain.
- An agent is a loop (observe, inspect, choose, act) around a model with tools, memory and environment feedback; ReAct (2022) formalized it as interleaved thought, action and observation.
- The harness often matters as much as the model: same model, different scaffolding, different product; vendors post-train for their harness.
- Six components of a coding harness: live repo context, stable-prefix prompt shape with prompt caching, structured tools with validation and permissions, context reduction, transcript plus working memory, bounded subagents.
- Prompt caching reuses the KV cache of an exact token prefix; writes cost about 1.25× and reads about 0.1× input price (Anthropic), so a 20-step run with a 20k prefix and 1k per step costs about 104k instead of 590k input-token units, about 5.7× less; put stable content first.
- Agents for ML research: autoresearch (edit one `train.py`, 5-minute runs, val_bpb, about 12 experiments an hour) and OpenResearch (literature, hypotheses, experiment ledger, remote compute); the agent is the instrument, the student is the scientist: seed-checked ablations, a human-written note, an individual oral.
- Lecture 22 takes the rest: tool protocols (MCP, A2A), memory across sessions, multi-agent orchestration, agentic RL in practice, evaluating agents, and where they fail.

## Further reading

**Reasoning models**

- [Raschka, *Understanding Reasoning LLMs* (2025)](https://magazine.sebastianraschka.com/p/understanding-reasoning-llms): the source of part 1.
- [DeepSeek-AI, *DeepSeek-R1* (2025)](https://arxiv.org/abs/2501.12948): sections 2 and 4 (including "unsuccessful attempts").
- [Snell et al., *Scaling LLM Test-Time Compute Optimally* (2024)](https://arxiv.org/abs/2408.03314).
- [Yang et al., *Qwen3 Technical Report* (2025)](https://arxiv.org/abs/2505.09388): the thinking budget and thinking-mode fusion.
- [Muennighoff et al., *s1: Simple test-time scaling* (2025)](https://arxiv.org/abs/2501.19393); [Sky-T1](https://novasky-ai.github.io/posts/sky-t1/); [TinyZero](https://github.com/Jiayi-Pan/TinyZero).

**Agents**

- [Yao et al., *ReAct* (2022)](https://arxiv.org/abs/2210.03629).
- [Raschka, *Components of a Coding Agent* (2026)](https://magazine.sebastianraschka.com/p/components-of-a-coding-agent) and the [Mini Coding Agent](https://github.com/rasbt/mini-coding-agent): read the code; it is short.
- [Anthropic, *Prompt caching*](https://platform.claude.com/docs/en/build-with-claude/prompt-caching) and [*Extended thinking*](https://platform.claude.com/en/docs/build-with-claude/extended-thinking).
- [karpathy/autoresearch](https://github.com/karpathy/autoresearch) and [alphaXiv/OpenResearch](https://github.com/alphaXiv/OpenResearch).

## Discussion questions

1. Snell et al. find that test-time compute loses to a bigger pretrained model on the hardest questions. Why would that be, mechanistically? What would a question need to look like for extra sampling to help?
2. R1-Zero is ahead of R1 on GPQA Diamond and behind on everything else. What does each SFT stage add, and which benchmark would you expect it to cost, and why?
3. "Harness > model" implies leaderboards that score a bare model miss most of the product. Design an evaluation that separates the harness's contribution from the model's.
4. Which of the six harness components would be hardest to carry over to a data-analysis or research agent, and what would replace it? Compare your answer with the OpenResearch table.
5. In an autoresearch run, the agent reports a 0.8% improvement in `val_bpb` from a change. List three ways that number could be wrong, and the cheapest experiment that rules out each.